In [ ]:
import time
from pyspark.sql import SparkSession
from pyspark.ml import PipelineModel
from pyspark.ml.regression import LinearRegression, GBTRegressor
from pyspark.ml.tuning import ParamGridBuilder, TrainValidationSplit
from pyspark.ml.evaluation import RegressionEvaluator
from pyspark.sql.functions import col

# 1. INITIALIZE SPARK
spark = SparkSession.builder \
    .appName("NYC_Taxi_Stage7_Regression") \
    .config("spark.driver.memory", "8g") \
    .config("spark.executor.memory", "8g") \
    .getOrCreate()
spark.sparkContext.setLogLevel("ERROR")

# 2. LOAD DATA AND PIPELINE
print("Loading data and Pipeline...")
train_df = spark.read.parquet("../data/processed/train.parquet")
val_df = spark.read.parquet("../data/processed/val.parquet")

# We unite train and val because TrainValidationSplit handles the internal split
training_data = train_df.union(val_df)

# CRITICAL DOMAIN LOGIC: The Regressor only predicts the amount IF a tip is given.
# Therefore, we train it ONLY on trips where has_tip == 1.
print("Filtering data for Regression (only trips with tips)...")
training_data_tips_only = training_data.filter(col("has_tip") == 1)

pipeline_model = PipelineModel.load("../models/preprocessing_pipeline")
print("Transforming data through the pipeline...")
prepared_data = pipeline_model.transform(training_data_tips_only)

# 3. DEFINE EVALUATOR (Rubric: Use appropriate metric. RMSE is standard for regression)
evaluator = RegressionEvaluator(
    labelCol="tip_amount", 
    predictionCol="prediction", 
    metricName="rmse" # Root Mean Squared Error
)

# *************************************
# MODEL 1: LINEAR REGRESSION (Baseline)
# *************************************
print("\n--- Training Model 1: Linear Regression ---")
lr = LinearRegression(featuresCol="features", labelCol="tip_amount")

# Search Space: 2 x 2 = 4 combinations
lr_paramGrid = (ParamGridBuilder()
    .addGrid(lr.regParam, [0.01, 0.1])
    .addGrid(lr.elasticNetParam, [0.0, 0.5])
    .build())

lr_tvs = TrainValidationSplit(
    estimator=lr,
    estimatorParamMaps=lr_paramGrid,
    evaluator=evaluator,
    trainRatio=0.8
)

start_time = time.time()
lr_model = lr_tvs.fit(prepared_data)
lr_duration = time.time() - start_time

print(f"Linear Regression Tuning completed in {lr_duration/60:.2f} minutes.")
print(f"Search space size: {len(lr_paramGrid)} combinations.")

best_lr = lr_model.bestModel
best_lr.write().overwrite().save("../models/best_linear_regression")

# **********************************************
# MODEL 2: GRADIENT-BOOSTED TREES (GBTRegressor)
# **********************************************
print("\n--- Training Model 2: GBT Regressor ---")
# GBTs are powerful sequential ensembles, excellent for complex non-linear patterns
gbt = GBTRegressor(featuresCol="features", labelCol="tip_amount")

# Search Space: 2 x 2 = 4 combinations
# GBT is computationally expensive, so we keep maxIter relatively low for tuning
gbt_paramGrid = (ParamGridBuilder()
    .addGrid(gbt.maxDepth, [5, 7])
    .addGrid(gbt.maxIter, [20, 40])
    .build())

gbt_tvs = TrainValidationSplit(
    estimator=gbt,
    estimatorParamMaps=gbt_paramGrid,
    evaluator=evaluator,
    trainRatio=0.8
)

start_time = time.time()
gbt_model = gbt_tvs.fit(prepared_data)
gbt_duration = time.time() - start_time

print(f"GBT Regression Tuning completed in {gbt_duration/60:.2f} minutes.")
print(f"Search space size: {len(gbt_paramGrid)} combinations.")

best_gbt = gbt_model.bestModel
best_gbt.write().overwrite().save("../models/best_gbt_regression")

print("\nStage 7 (Regression) completed successfully. Best models saved to disk!")

26/05/27 02:20:33 WARN Utils: Your hostname, LaptopPablo resolves to a loopback address: 127.0.1.1; using 10.255.255.254 instead (on interface lo)
26/05/27 02:20:33 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address


Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).


26/05/27 02:20:34 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


Loading data and Pipeline...


Filtering data for Regression (only trips with tips)...


...


[Stage 1722:======>                                               (4 + 16) / 33]




[Stage 1722:================================>                    (20 + 13) / 33]




[Stage 1732:========>                                             (5 + 16) / 33]



GBT Regression Tuning completed in 5.87 minutes.
Search space size: 4 combinations.



Stage 7 (Regression) completed successfully. Best models saved to disk!
